# EDA part

In [1]:
import csv
import pandas as pd

# Column names in order, taken from docs/RCL.txt (fields 1-29)
COLUMNS = [
    "RECORD_ID", "CAMPNO", "MAKETXT", "MODELTXT", "YEARTXT", "MFGCAMPNO",
    "COMPNAME", "MFGNAME", "BGMAN", "ENDMAN", "RCLTYPECD", "POTAFF",
    "ODATE", "INFLUENCED_BY", "MFGTXT", "RCDATE", "DATEA", "RPNO", "FMVSS",
    "DESC_DEFECT", "CONEQUENCE_DEFECT", "CORRECTIVE_ACTION", "NOTES",
    "RCL_CMPT_ID", "MFR_COMP_NAME", "MFR_COMP_DESC", "MFR_COMP_PTNO",
    "DO_NOT_DRIVE", "PARK_OUTSIDE",
]
len(COLUMNS)

29

In [2]:
df = pd.read_csv(
    "../data/raw/FLAT_RCL_POST_2010.txt",
    sep="\t",                  # file is tab-delimited
    header=None,               # no header row in the file
    names=COLUMNS,             # so we supply the names ourselves
    dtype=str,                 # read everything as text first; convert types later
    encoding="latin-1",        # safe default for older government text files
    quoting=csv.QUOTE_NONE,    # treat quote characters as normal text
    on_bad_lines="warn",       # report malformed rows instead of crashing
)
df.shape

(246899, 29)

In [5]:
df.head(3)

,RECORD_ID,CAMPNO,MAKETXT,MODELTXT,YEARTXT,MFGCAMPNO,COMPNAME,MFGNAME,BGMAN,ENDMAN,...,DESC_DEFECT,CONEQUENCE_DEFECT,CORRECTIVE_ACTION,NOTES,RCL_CMPT_ID,MFR_COMP_NAME,MFR_COMP_DESC,MFR_COMP_PTNO,DO_NOT_DRIVE,PARK_OUTSIDE
0,81716,10V143000,GLAVAL,TITAN,2006,NaN,EQUIPMENT ADAPTIVE/MOBILITY,GLAVAL BUS DIVISION,20071201,20091231,...,GLAVAL BUS IS RECALLING CERTAIN MODEL YEAR 200...,THE USER OF THE LIFT COULD BE INJURED SHOULD T...,GLAVAL BUS IS WORKING WITH RICON AND RICON WIL...,OWNERS MAY ALSO CONTACT THE NATIONAL HIGHWAY T...,000035678000216461000001004,NaN,NaN,NaN,No,No
1,81717,10V080000,GEM,eS,2009,NaN,PARKING BRAKE,CHRYSLER GROUP GLOBAL ELECTRIC MOTORCARS,20091208,20091215,...,GEM IS RECALLING CERTAIN MODEL YEAR 2009 NEIGH...,THE VEHICLE COULD ROLL AWAY ON A HILL CREATING...,DEALERS WILL REPLACE THE PARKING BRAKE LEVER A...,OWNERS MAY ALSO CONTACT THE NATIONAL HIGHWAY T...,000035299000847927000000115,NaN,NaN,NaN,No,No
2,81718,10V419000,NU WA,DISCOVERY SERIES,2010,NaN,EQUIPMENT,"NU WA INDUSTRIES, INC.",NaN,NaN,...,NU WA IS RECALLING CERTAIN MODEL YEAR 2000-201...,AN OVERHEATED RECEIVER COULD CAUSE A FIRE.,NU WA IS WORKING WITH DIMPLEX AND DIMPLEX WILL...,OWNERS MAY ALSO CONTACT THE NATIONAL HIGHWAY T...,000037358000962769000000329,NaN,NaN,NaN,No,No


In [6]:
print("Rows:", len(df))
print("Unique recalls (CAMPNO):", df["CAMPNO"].nunique())
print("RCDATE range:", df["RCDATE"].min(), "to", df["RCDATE"].max())
print()
print(df["RCLTYPECD"].value_counts())

Rows: 246899
Unique recalls (CAMPNO): 15355
RCDATE range: 20100101 to 20261021

RCLTYPECD
V    220165
T     20282
E      5992
C       460
Name: count, dtype: int64


In [7]:
# Does POTAFF repeat within the same recall?
potaff_per_recall = df.groupby("CAMPNO")["POTAFF"].nunique()
print("Recalls with exactly one POTAFF value:", (potaff_per_recall == 1).mean().round(3))

Recalls with exactly one POTAFF value: 1.0


In [8]:
# Share of missing values per column (0.25 = 25% missing)
df.isna().mean().round(3).sort_values(ascending=False)

RPNO                 0.993
FMVSS                0.722
BGMAN                0.663
ENDMAN               0.663
MFR_COMP_PTNO        0.312
MFR_COMP_DESC        0.302
MFR_COMP_NAME        0.287
MFGCAMPNO            0.264
NOTES                0.133
ODATE                0.018
MODELTXT             0.000
YEARTXT              0.000
RECORD_ID            0.000
POTAFF               0.000
RCLTYPECD            0.000
COMPNAME             0.000
MFGNAME              0.000
CAMPNO               0.000
MAKETXT              0.000
INFLUENCED_BY        0.000
MFGTXT               0.000
CONEQUENCE_DEFECT    0.000
DESC_DEFECT          0.000
RCDATE               0.000
DATEA                0.000
RCL_CMPT_ID          0.000
CORRECTIVE_ACTION    0.000
DO_NOT_DRIVE         0.000
PARK_OUTSIDE         0.000
dtype: float64

In [12]:
# 1) Is PARK_OUTSIDE constant within each recall (like POTAFF)?
print("Recalls with one PARK_OUTSIDE value:",
      (df.groupby("CAMPNO")["PARK_OUTSIDE"].nunique() == 1).mean().round(3))

# 2) Count recalls (not rows): keep one row per CAMPNO
camp = df.drop_duplicates("CAMPNO")
print("Recall-level rows:", len(camp))
pd.crosstab(camp["RC_YEAR"], camp["PARK_OUTSIDE"])

Recalls with one PARK_OUTSIDE value: 1.0
Recall-level rows: 15355


PARK_OUTSIDE,No,Yes
RC_YEAR,,
2010,722,0
2011,655,0
2012,661,0
2013,710,0
2014,868,0
2015,970,1
2016,1031,0
2017,897,0
2018,1032,1
